AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors.

In [1]:
import pandas as pd
from pathlib import Path

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

PRELIM_TRAIN_PATH = PROCESSED_DIR / 'omni_train_prelim.parquet'
TRAIN_PATH = PROCESSED_DIR / 'omni_train.parquet'
VALIDATION_PATH = PROCESSED_DIR / 'omni_validation.parquet'

# I. Load Train Pool

In [2]:
df_train_full = pd.read_parquet(PRELIM_TRAIN_PATH)

print(f'Shape: {df_train_full.shape}')
print(f'Datetime range: {df_train_full["datetime"].min()} through {df_train_full["datetime"].max()}')
print(df_train_full.dtypes)

assert df_train_full['datetime'].is_monotonic_increasing, 'omni_train.parquet is not sorted by datetime'
print('Confirmed: datetime column is monotonically increasing')

Shape: (12358079, 19)
Datetime range: 2000-01-01 00:00:00 through 2023-06-30 23:58:00
datetime               datetime64[us]
year                            int64
day                             int64
hour                            int64
minute                          int64
mag_avg_nt                    float64
bx_gsm_nt                     float64
by_gsm_nt                     float64
bz_gsm_nt                     float64
flow_speed_km_s               float64
proton_density_n_cc           float64
day_cos                       float64
day_sin                       float64
hour_cos                      float64
hour_sin                      float64
minute_cos                    float64
minute_sin                    float64
kp_10                           int64
data_split                     object
dtype: object
Confirmed: datetime column is monotonically increasing


# II. Candidate Validation Cutoffs

In [3]:
# Exploratory cell to consider different cut-offs
candidate_cutoffs = ['2020-01-01', '2021-01-01']
total_rows = len(df_train_full)

for cutoff in candidate_cutoffs:
    cutoff_ts = pd.Timestamp(cutoff)
    final_train_rows = (df_train_full['datetime'] < cutoff_ts).sum()
    validation_rows = total_rows - final_train_rows
    print(f'Cutoff {cutoff}: final_train={final_train_rows:,} ({100 * final_train_rows / total_rows:.2f}%), '
          f'validation={validation_rows:,} ({100 * validation_rows / total_rows:.2f}%)')

Cutoff 2020-01-01: final_train=10,519,200 (85.12%), validation=1,838,879 (14.88%)
Cutoff 2021-01-01: final_train=11,046,240 (89.38%), validation=1,311,839 (10.62%)


In [4]:
VALIDATION_CUTOFF = pd.Timestamp('2020-01-01 00:00:00')

assert (VALIDATION_CUTOFF.month, VALIDATION_CUTOFF.day, VALIDATION_CUTOFF.hour, VALIDATION_CUTOFF.minute) == (1, 1, 0, 0), \
    'VALIDATION_CUTOFF must fall on a Jan 1 00:00:00 whole-year boundary'
print(f'Validation cutoff selected: {VALIDATION_CUTOFF} (whole-year boundary confirmed)')

Validation cutoff selected: 2020-01-01 00:00:00 (whole-year boundary confirmed)


# III. Train (Final) / Validation Split

In [ ]:
train_mask = df_train_full['datetime'] < VALIDATION_CUTOFF

df_train_final = df_train_full.loc[train_mask].copy()
df_validation = df_train_full.loc[~train_mask].copy()
df_validation['data_split'] = 'validation'

print(f'df_train_final rows: {len(df_train_final):,}')
print(f'df_validation rows: {len(df_validation):,}')

In [6]:
original_rows = len(df_train_full)
train_final_rows = len(df_train_final)
validation_rows = len(df_validation)
combined_rows = train_final_rows + validation_rows

print(f'Original omni_train.parquet rows: {original_rows:,}')
print(f'Final train rows: {train_final_rows:,}')
print(f'Validation rows: {validation_rows:,}')
print(f'Combined rows: {combined_rows:,}')
print(f'All rows preserved: {original_rows == combined_rows}')
print(f'Validation percentage: {100 * validation_rows / original_rows:.2f}%')

assert original_rows == combined_rows, 'Row count mismatch after split'

Original omni_train.parquet rows: 12,358,079
Final train rows: 10,519,200
Validation rows: 1,838,879
Combined rows: 12,358,079
All rows preserved: True
Validation percentage: 14.88%


In [7]:
for label, df in [('Train (final)', df_train_final), ('Validation', df_validation)]:
    print(f'{label}: {df["datetime"].min()} through {df["datetime"].max()} ({len(df):,} rows)')

Train (final): 2000-01-01 00:00:00 through 2019-12-31 23:59:00 (10,519,200 rows)
Validation: 2020-01-01 00:00:00 through 2023-06-30 23:58:00 (1,838,879 rows)


In [8]:
print('df_train_final data_split value counts:')
print(df_train_final['data_split'].value_counts())
print()
print('df_validation data_split value counts:')
print(df_validation['data_split'].value_counts())

df_train_final data_split value counts:


data_split
train    10519200
Name: count, dtype: int64



df_validation data_split value counts:


data_split
validation    1838879
Name: count, dtype: int64


In [9]:
years_train_final = set(df_train_final['year'].unique())
years_validation = set(df_validation['year'].unique())
years_overlap = years_train_final & years_validation

assert not years_overlap, f'Whole-year boundary violated for years: {sorted(years_overlap)}'
print('Whole-year boundary confirmed: no calendar year is split across train_final and validation.')
print(f'Train (final) years: {sorted(years_train_final)}')
print(f'Validation years: {sorted(years_validation)}')
print('Note: 2023 appears only under validation, as a partial (Jan-Jun) year - '
      'inherited unchanged from the pre-existing train/test cut made in notebook 02, '
      'not a new mid-year split introduced here.')

Whole-year boundary confirmed: no calendar year is split across train_final and validation.
Train (final) years: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Validation years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Note: 2023 appears only under validation, as a partial (Jan-Jun) year - inherited unchanged from the pre-existing train/test cut made in notebook 02, not a new mid-year split introduced here.


# IV. Write Outputs

In [10]:
df_train_final.to_parquet(TRAIN_PATH)
df_validation.to_parquet(VALIDATION_PATH)

print(f'Wrote {len(df_train_final):,} rows to {TRAIN_PATH.as_posix()}')
print(f'Wrote {len(df_validation):,} rows to {VALIDATION_PATH.as_posix()}')

Wrote 10,519,200 rows to work/Processed/omni_train.parquet
Wrote 1,838,879 rows to work/Processed/omni_validation.parquet


In [11]:
df_train_final_reloaded = pd.read_parquet(TRAIN_PATH)
df_validation_reloaded = pd.read_parquet(VALIDATION_PATH)

assert len(df_train_final_reloaded) == len(df_train_final)
assert len(df_validation_reloaded) == len(df_validation)
assert df_train_final_reloaded['datetime'].min() == df_train_final['datetime'].min()
assert df_train_final_reloaded['datetime'].max() == df_train_final['datetime'].max()
assert df_validation_reloaded['datetime'].min() == df_validation['datetime'].min()
assert df_validation_reloaded['datetime'].max() == df_validation['datetime'].max()

print('Reload sanity check passed - row counts and date ranges match what was written.')
print(f'omni_train.parquet: {len(df_train_final_reloaded):,} rows, '
      f'{df_train_final_reloaded["datetime"].min()} through {df_train_final_reloaded["datetime"].max()}')
print(f'omni_validation.parquet: {len(df_validation_reloaded):,} rows, '
      f'{df_validation_reloaded["datetime"].min()} through {df_validation_reloaded["datetime"].max()}')

Reload sanity check passed - row counts and date ranges match what was written.
omni_train.parquet: 10,519,200 rows, 2000-01-01 00:00:00 through 2019-12-31 23:59:00
omni_validation.parquet: 1,838,879 rows, 2020-01-01 00:00:00 through 2023-06-30 23:58:00


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>